In [1]:
from google.colab import files

uploaded = files.upload()

Saving test_fe.csv to test_fe.csv
Saving train_fe.csv to train_fe.csv


In [2]:
from google.colab import files

uploaded = files.upload()

Saving sample_submission.csv to sample_submission.csv


In [3]:
!pip install -q optuna xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 16.9 MB/s eta 0:00:00


### Install Libraries

In [4]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score

from xgboost import XGBClassifier

### Import Libraries and Setup

In [5]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


### Check GPU Availability

In [6]:
train = pd.read_csv("train_fe.csv")
test = pd.read_csv("test_fe.csv")

sample_submission = pd.read_csv(
    "sample_submission.csv"
)

print(train.shape)
print(test.shape)

(577347, 19)
(247435, 18)


### Load Data

In [7]:
for df in [train, test]:

    df["alpha_rad"] = np.radians(df["alpha"])
    df["delta_rad"] = np.radians(df["delta"])

    df["x_coord"] = (
        np.cos(df["delta_rad"])
        * np.cos(df["alpha_rad"])
    )

    df["y_coord"] = (
        np.cos(df["delta_rad"])
        * np.sin(df["alpha_rad"])
    )

    df["z_coord"] = np.sin(
        df["delta_rad"]
    )

    df["u_div_g"] = (
        df["u"] /
        (df["g"] + 1e-6)
    )

    df["g_div_r"] = (
        df["g"] /
        (df["r"] + 1e-6)
    )

    df["r_div_i"] = (
        df["r"] /
        (df["i"] + 1e-6)
    )

    df["i_div_z"] = (
        df["i"] /
        (df["z"] + 1e-6)
    )

    df["redshift_sq"] = (
        df["redshift"] ** 2
    )

    df["redshift_cube"] = (
        df["redshift"] ** 3
    )

### Feature Engineering - Coordinate Transformations and Ratios

In [8]:
for df in [train, test]:

    # Trigonometric sky features

    df["sin_alpha"] = np.sin(
        np.radians(df["alpha"])
    )

    df["cos_alpha"] = np.cos(
        np.radians(df["alpha"])
    )

    df["sin_delta"] = np.sin(
        np.radians(df["delta"])
    )

    df["cos_delta"] = np.cos(
        np.radians(df["delta"])
    )

    # Strong feature interactions

    df["redshift_griz"] = (
        df["redshift"]
        * df["griz"]
    )

    df["redshift_ugri"] = (
        df["redshift"]
        * df["ugri"]
    )

    df["redshift_u_g"] = (
        df["redshift"]
        * df["u_g"]
    )

    df["redshift_g_r"] = (
        df["redshift"]
        * df["g_r"]
    )

    df["redshift_r_i"] = (
        df["redshift"]
        * df["r_i"]
    )

    df["redshift_sq_griz"] = (
        df["redshift_sq"]
        * df["griz"]
    )

    df["alpha_delta"] = (
        df["alpha"]
        * df["delta"]
    )

    df["alpha_redshift"] = (
        df["alpha"]
        * df["redshift"]
    )

    df["delta_redshift"] = (
        df["delta"]
        * df["redshift"]
    )

### Feature Engineering - Trigonometric and Interaction Features

In [9]:
all_redshift = pd.concat([
    train["redshift"],
    test["redshift"]
])

bins = pd.qcut(
    all_redshift,
    q=20,
    duplicates="drop",
    retbins=True
)[1]

for df in [train, test]:

    df["redshift_bin"] = pd.cut(
        df["redshift"],
        bins=bins,
        labels=False,
        include_lowest=True
    )

### Feature Engineering - Redshift Binning

In [10]:
for col in [
    "spectral_type",
    "galaxy_population"
]:

    le = LabelEncoder()

    combined = pd.concat([
        train[col],
        test[col]
    ])

    le.fit(combined)

    train[col] = le.transform(
        train[col]
    )

    test[col] = le.transform(
        test[col]
    )

### Label Encoding Categorical Features

In [11]:
TARGET = "class"

X = train.drop(
    columns=["id", TARGET]
)

X_test = test.drop(
    columns=["id"]
)

target_encoder = LabelEncoder()

y = target_encoder.fit_transform(
    train[TARGET]
)

print(X.shape)

(577347, 42)


### Prepare Data for Model Training

In [12]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = []

test_probs = np.zeros(
    (
        len(test),
        len(target_encoder.classes_)
    )
)

### Setup Stratified K-Fold Cross-Validation

In [45]:
for fold, (
    train_idx,
    valid_idx
) in enumerate(
    skf.split(X, y)
):

    print(
        f"\nFold {fold+1}"
    )

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y[train_idx]
    y_valid = y[valid_idx]

    model = XGBClassifier(

    objective="multi:softprob",

    num_class=3,

    n_estimators=2500,

    learning_rate=0.02,

    max_depth=10,

    subsample=0.8,

    colsample_bytree=0.8,

    min_child_weight=3,

    reg_alpha=0.1,

    reg_lambda=1.0,

    random_state=42,

    tree_method="hist",

    device="cuda",

    eval_metric="mlogloss"
)

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(
        X_valid
    )

    score = balanced_accuracy_score(
        y_valid,
        pred
    )

    cv_scores.append(score)

    print(score)

    test_probs += (
        model.predict_proba(
            X_test
        )
        / skf.n_splits
    )


Fold 1


KeyboardInterrupt: 

### Model Training and Cross-Validation

In [37]:
print(
    "Mean CV:",
    np.mean(cv_scores)
)

Mean CV: 0.9579640267072409


### Display Mean Cross-Validation Score

In [38]:
feature_importance = pd.DataFrame({

    "feature":
        X.columns,

    "importance":
        model.feature_importances_

})

feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False
)

feature_importance.head(30)

,feature,importance
27,redshift_cube,0.298925
16,redshift_log,0.141808
7,redshift,0.141176
15,griz,0.097769
26,redshift_sq,0.062786
37,redshift_sq_griz,0.055987
14,ugri,0.029943
41,redshift_bin,0.014330
3,g,0.013542
23,g_div_r,0.012673


### Feature Importance Analysis

In [39]:
feature_importance.to_csv(
    "xgb_c_importance.csv",
    index=False
)

print("Saved: xgb_c_importance.csv")

Saved: xgb_c_importance.csv


### Save Feature Importance

In [40]:
pd.DataFrame({

    "fold": range(
        1,
        len(cv_scores)+1
    ),

    "score": cv_scores

}).to_csv(
    "xgb_c_cv_scores.csv",
    index=False
)

print("Saved: xgb_c_cv_scores.csv")

Saved: xgb_c_cv_scores.csv


### Save Cross-Validation Scores

In [41]:
preds = np.argmax(
    test_probs,
    axis=1
)

submission = pd.DataFrame({

    "id": test["id"],

    "class":
        target_encoder.inverse_transform(
            preds
        )
})

submission.to_csv(
    "submission_xgb_c.csv",
    index=False
)

print("Saved: submission_xgb_c.csv")

Saved: submission_xgb_c.csv


### Generate Submission File

In [42]:
prob_df = pd.DataFrame(
    test_probs,
    columns=target_encoder.classes_
)

prob_df.insert(
    0,
    "id",
    test["id"]
)

prob_df.to_csv(
    "xgb_c_prob.csv",
    index=False
)

print("Saved: xgb_c_prob.csv")

Saved: xgb_c_prob.csv


### Save Prediction Probabilities

In [43]:
cv_summary = pd.DataFrame({

    "mean_cv": [
        np.mean(cv_scores)
    ],

    "std_cv": [
        np.std(cv_scores)
    ]

})

cv_summary.to_csv(
    "xgb_c_cv_summary.csv",
    index=False
)

cv_summary

,mean_cv,std_cv
0,0.957964,0.000734


### Cross-Validation Summary

In [44]:
from google.colab import files

files.download(
    "submission_xgb_c.csv"
)

files.download(
    "xgb_c_prob.csv"
)

files.download(
    "xgb_c_importance.csv"
)

files.download(
    "xgb_c_cv_scores.csv"
)

files.download(
    "xgb_c_cv_summary.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>